# Euro Area Industrial Production Analysis

## Time Series Decomposition, Seasonal Adjustment and Forecasting Preparation

This notebook presents a complete time series preprocessing workflow using monthly Industrial Production Index data retrieved directly from the Eurostat API.

The analysis covers:

- automated data acquisition,
- exploratory time series analysis,
- seasonal decomposition,
- seasonal adjustment,
- stationarity testing,
- outlier detection and treatment,
- and candidate ARIMA model selection.

The objective is to transform raw macroeconomic data into a clean and stationary time series suitable for forecasting and econometric modelling.

## Project Workflow

The analysis follows the following pipeline:

1. Load required packages
2. Retrieve data from Eurostat
3. Prepare the time series
4. Exploratory Data Analysis
5. Trend and seasonality identification
6. Seasonal adjustment
7. Stationarity testing
8. Outlier detection
9. Outlier treatment
10. ARIMA model selection
11. Conclusions

# Setup

The following packages are used throughout the analysis for data retrieval, visualization, time series decomposition, statistical testing and forecasting.

In [ ]:
# ----- 0. Install / load packages -----
if (!requireNamespace("eurostat", quietly = TRUE))  install.packages("eurostat")
if (!requireNamespace("tseries",  quietly = TRUE))  install.packages("tseries")
if (!requireNamespace("forecast", quietly = TRUE))  install.packages("forecast")
if (!requireNamespace("seasonal", quietly = TRUE))  install.packages("seasonal")
if (!requireNamespace("ggplot2",  quietly = TRUE))  install.packages("ggplot2")
if (!requireNamespace("dplyr",    quietly = TRUE))  install.packages("dplyr")
if (!requireNamespace("tsoutliers",quietly = TRUE)) install.packages("tsoutliers")

library(eurostat)
library(tseries)
library(forecast)
library(seasonal)
library(ggplot2)
library(dplyr)

# 1. Data Acquisition

This project retrieves the Industrial Production Index directly from the Eurostat API using the `eurostat` package.

Using the official API ensures that the analysis is fully reproducible without relying on locally stored datasets.

In [ ]:
# ----- 1. Data Selection -----
# Dataset: Monthly Industrial Production Index for the Euro Area (EA20)
# Source : Eurostat — sts_inpr_m
# Filters: nace_r2 = "B-D"  (total industry excl. construction)
#          s_adj   = "NSA"   (not seasonally adjusted — raw data)
#          unit    = "I15"   (index, 2015=100)
#          geo     = "EA20"

raw <- get_eurostat("sts_inpr_m",
                    filters = list(
                      nace_r2 = "B-D",
                      s_adj   = "NSA",
                      unit    = "I15",
                      geo     = "EA20"
                    ),
                    time_format = "date")

# Keep 2000-01 to 2023-12 for a balanced 24-year window
df <- raw %>%
  filter(time >= as.Date("2000-01-01"),
         time <= as.Date("2023-12-01")) %>%
  arrange(time)

The resulting dataset contains monthly observations of the Euro Area Industrial Production Index from January 2000 to December 2023.

The series is retrieved in its raw (not seasonally adjusted) form so that all preprocessing steps can be performed explicitly within the analysis.

# 2. Data Preparation

The raw data retrieved from Eurostat require a small number of preprocessing steps before the statistical analysis can begin.

This stage prepares the monthly time series, ensures that dates are correctly formatted, and creates the final object used throughout the remainder of the notebook.

In [ ]:
# Convert to monthly ts object
ipi_ts <- ts(df$values, start = c(2000, 1), frequency = 12)

At this point, the data have been transformed into a monthly time series that serves as the input for all subsequent analyses.

# 3. Exploratory Data Analysis

Before applying any statistical procedures, the raw Industrial Production Index is explored visually and numerically.

The objective is to understand its overall behaviour, identify potential structural changes, and obtain an initial impression of its distribution.

In [ ]:
# ----- 2. Descriptive Analysis -----
cat("=== Descriptive Statistics ===\n")
print(summary(ipi_ts))
cat("Observations :", length(ipi_ts), "\n")
cat("Std. Dev.    :", round(sd(ipi_ts), 3), "\n")
cat("Skewness     :", round(moments::skewness(ipi_ts), 3), "\n")
cat("Kurtosis     :", round(moments::kurtosis(ipi_ts), 3), "\n")

The descriptive statistics provide a first indication of the variability and distribution of the Industrial Production Index over the sample period.

In [ ]:
# Time-series plot
autoplot(ipi_ts) +
  labs(title = "Euro Area Industrial Production Index (2000–2023)",
       subtitle = "Monthly, NSA, 2015=100",
       x = "Date", y = "Index") +
  theme_minimal()
ggsave("fig1_raw_series.png", width = 9, height = 4)

### Interpretation

The series exhibits several distinct phases, including the Global Financial Crisis, the COVID-19 contraction and the post-pandemic recovery.

A clear seasonal pattern is also visible, suggesting that seasonal adjustment will be required before modelling.

# 4. Feature Identification

This section investigates the main statistical properties of the Industrial Production Index before any transformation is applied.

Three complementary approaches are used:

- the Autocorrelation Function (ACF),
- the Partial Autocorrelation Function (PACF),
- and classical time series decomposition.

Together, these diagnostics provide evidence regarding trend, persistence and seasonality, guiding the subsequent modelling strategy.

## 4.1 Autocorrelation Analysis

The autocorrelation structure is examined using both the ACF and PACF.

These plots help identify persistence in the series and provide preliminary evidence regarding autoregressive and seasonal dynamics.

In [ ]:
# ----- 3. Feature Identification -----
# Lag/ACF/PACF
png("fig2_acf_pacf.png", width = 900, height = 400)
par(mfrow = c(1,2))
acf(ipi_ts,  lag.max = 48, main = "ACF — Raw IPI")
pacf(ipi_ts, lag.max = 48, main = "PACF — Raw IPI")
dev.off()

### Interpretation

The ACF displays a slow decay together with pronounced peaks at multiples of twelve months, indicating both persistence and strong annual seasonality.

The PACF shows a dominant spike at lag 1 together with smaller seasonal spikes, suggesting that autoregressive dynamics are present in the monthly series.

## 4.2 Classical Time Series Decomposition

To further investigate the underlying structure of the series, a classical multiplicative decomposition is performed.

This separates the observed values into trend, seasonal and irregular components, providing a clearer understanding of the data-generating process.

In [ ]:
# Classical decomposition (multiplicative — index data with proportional variation)
decomp_mult <- decompose(ipi_ts, type = "multiplicative")

In [ ]:
png("fig3_decomposition.png", width = 900, height = 700)
plot(decomp_mult)
dev.off()

### Interpretation

The decomposition confirms the presence of a persistent long-run trend together with a highly regular annual seasonal pattern.

Most irregular movements are associated with major economic events such as the Global Financial Crisis and the COVID-19 pandemic.

These findings support the need for explicit seasonal adjustment before proceeding to statistical modelling.

# 5. Seasonal Adjustment

Seasonal adjustment removes predictable calendar-related fluctuations from a time series, allowing the underlying economic movements to be analysed more clearly.

For this project, seasonal adjustment is performed using the X-13ARIMA-SEATS procedure, which is the official methodology adopted by Eurostat and many national statistical agencies.

## Why X-13ARIMA-SEATS?

The X-13ARIMA-SEATS procedure was selected because it:

- adjusts for calendar and trading-day effects,
- accounts for movable holidays such as Easter,
- provides statistically validated seasonal factors,
- is widely regarded as the benchmark approach for official macroeconomic statistics.

Compared with simpler decomposition methods, X-13 produces seasonally adjusted series that are more suitable for forecasting and econometric modelling.

In [ ]:
# ----- 4. Seasonal Adjustment — X-13ARIMA-SEATS -----
x13_model <- seas(ipi_ts)
summary(x13_model)

ipi_sa  <- final(x13_model)          # seasonally adjusted series
ipi_irr <- irregular(x13_model)      # irregular component

In [ ]:
autoplot(cbind(Raw = ipi_ts, `SA (X-13)` = ipi_sa)) +
  labs(title = "IPI: Raw vs. Seasonally Adjusted",
       x = "Date", y = "Index") +
  theme_minimal()
ggsave("fig4_sa_comparison.png", width = 9, height = 4)

In [ ]:
# Seasonal subseries plot (verify removal)
png("fig5_seasonal_subseries.png", width = 900, height = 500)
monthplot(ipi_sa, main = "Seasonal Sub-series — SA Series")
dev.off()

### Interpretation

The seasonally adjusted series removes the recurring annual pattern while preserving the underlying trend-cycle.

The seasonal subseries plot indicates that the systematic monthly variation has largely disappeared, suggesting that the adjustment has been successful and that the resulting series is appropriate for further statistical analysis.

# 6. Stationarity Analysis

Many time series models require stationarity, meaning that the statistical properties of the series remain stable over time.

This section evaluates whether the Industrial Production Index satisfies this assumption before forecasting models are estimated.

## 6.1 Unit Root Tests

In [ ]:
# ----- 5. Stationarity Tests -----
# 5a. ADF test on raw series
cat("\n=== ADF Test — Raw Series ===\n")
print(adf.test(ipi_ts))

# 5b. ADF on SA series
cat("\n=== ADF Test — SA Series ===\n")
print(adf.test(ipi_sa))

# 5c. KPSS test
cat("\n=== KPSS Test — SA Series ===\n")
print(kpss.test(ipi_sa))

### Interpretation

Both statistical tests indicate that the seasonally adjusted Industrial Production Index is non-stationary.

The ADF test fails to reject the unit root hypothesis, while the KPSS test rejects level stationarity, providing consistent evidence that the series contains a stochastic trend.

## 6.2 First Differencing

In [ ]:
# 5d. First difference of SA series
ipi_d1 <- diff(ipi_sa)

In [ ]:
autoplot(diff(ipi_sa)) +
  labs(title = "First-Differenced SA Industrial Production",
       x = "Date", y = "Δ Index") +
  theme_minimal()
ggsave("fig6_differenced.png", width = 9, height = 4)

In [ ]:
cat("\n=== ADF Test — First-Differenced SA ===\n")
print(adf.test(diff(ipi_sa)))
cat("\n=== KPSS Test — First-Differenced SA ===\n")
print(kpss.test(diff(ipi_sa)))

### Interpretation

After first differencing, both statistical tests indicate that the transformed series is stationary.

The differenced series fluctuates around a stable mean with approximately constant variance, confirming that the Industrial Production Index is integrated of order one, I(1).

# 7. Outlier Management

Extreme observations may distort model estimation and affect forecasting performance.

This section identifies unusual events in the Industrial Production Index and evaluates whether these observations reflect genuine economic shocks or statistical irregularities.

## 7.1 Outlier Detection

Outliers are detected using automated procedures designed specifically for time series analysis.

The objective is to identify additive outliers, level shifts and temporary changes that may influence model estimation.

In [ ]:
# ----- 6. Outlier Detection & Treatment -----
# Use tsoutliers / auto.arima + outlier detection
ipi_arima <- auto.arima(ipi_sa, stepwise = FALSE)
outliers  <- tsoutliers::tso(ipi_sa, types = c("AO","TC","LS","IO"))

In [ ]:
print(outliers)

In [ ]:
# Plot outlier-cleaned series
png("fig7_outliers.png", width = 900, height = 450)
plot(outliers)
dev.off()

### Interpretation

Several detected outliers correspond to major macroeconomic events, including the Global Financial Crisis and the COVID-19 pandemic.

These observations represent economically meaningful shocks rather than data errors and should therefore be interpreted carefully before any treatment is applied.

## 7.2 Outlier Treatment

To assess the sensitivity of subsequent analyses, an outlier-adjusted version of the series is constructed using automated cleaning procedures.

In [ ]:
ipi_clean <- tsclean(ipi_sa)   # fast robust cleaning via STL + Winsorisation
autoplot(cbind(`SA` = ipi_sa, `SA + Cleaned` = ipi_clean)) +
  labs(title = "SA IPI: Before and After Outlier Treatment",
       x = "Date", y = "Index") +
  theme_minimal()
ggsave("fig8_cleaned.png", width = 9, height = 4)

### Interpretation

The adjusted series preserves the overall economic dynamics while reducing the influence of extreme temporary disturbances.

This improves the robustness of statistical modelling and facilitates the identification of underlying patterns.

# 8. Model Selection

After seasonal adjustment, stationarity transformation and outlier treatment, candidate forecasting models can be evaluated.

Autocorrelation diagnostics and information criteria are used to identify appropriate ARIMA specifications.

## 8.1 Autocorrelation Diagnostics

In [ ]:
# ----- 7. Model Selection -----
# After differencing and cleaning, check ACF/PACF for candidate models
ipi_final <- diff(ipi_clean)

png("fig9_final_acf.png", width = 900, height = 400)
par(mfrow = c(1, 2))
acf(ipi_final,  lag.max = 36, main = "ACF — Final Processed Series")
pacf(ipi_final, lag.max = 36, main = "PACF — Final Processed Series")
dev.off()

The autocorrelation structure suggests that both autoregressive and moving average components may be required to model the series adequately.

## 8.2 Automatic ARIMA Selection

In [ ]:
# Auto-select ARIMA
best_model <- auto.arima(ipi_clean, d = 1, stepwise = FALSE,
                          approximation = FALSE, trace = TRUE)

In [ ]:
cat("\n=== Best ARIMA Model ===\n")
print(summary(best_model))

The selected model balances goodness-of-fit and parsimony according to standard information criteria.

## 8.3 Diagnostic Checking

In [ ]:
# Ljung-Box residual test
cat("\n=== Ljung-Box Test on Residuals ===\n")
print(Box.test(residuals(best_model), lag = 24, type = "Ljung-Box"))

# Residual diagnostics
png("fig10_residuals.png", width = 900, height = 600)
checkresiduals(best_model)
dev.off()

The residual diagnostics suggest that the selected model captures most of the systematic variation in the series.

Remaining residuals resemble white noise, supporting the adequacy of the specification.

# 9. Conclusions

This project demonstrated a complete workflow for macroeconomic time series analysis using Industrial Production Index data retrieved directly from the Eurostat API.

The analysis identified strong seasonality, non-stationarity and several economically meaningful outliers.

After seasonal adjustment and differencing, the series became suitable for statistical modelling.

The project illustrates an end-to-end analytical process, including:

- automated data acquisition,
- data preparation,
- exploratory analysis,
- seasonal adjustment,
- stationarity testing,
- outlier management,
- and model selection.

The resulting framework can serve as a foundation for forecasting applications and economic monitoring systems.